# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane: Classification<br>
Why?: To classify whether page needs refresh or not. This greatly reduces manual effort to look at analytics of every page.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy: 'decline_score'**<br>
This proxy label will be used to identify whether **'refresh_needed'** or not<br>
It is defined by the rule applied on 90 day latest window.<br>
90 days - compare prev30 and last30 to measure the observed signals.<br>



Here 'fact_content_query_90d' is used and only important and few features are being used to get observed signals, later on i will use warehouse dataset in the notebooks (except this notebook and previous)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
import numpy as np

In [ ]:
data = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d")

In [ ]:
sample_data = pd.DataFrame(data['train'])

In [ ]:
print(sample_data.describe())

       query_char_count  query_token_count  impressions_90d    clicks_90d  \
count      2.414248e+06       2.414248e+06     2.414248e+06  2.414248e+06   
mean       3.224091e+01       5.384297e+00     8.770558e+01  1.905016e-01   
std        2.662306e+01       4.277630e+00     9.332719e+02  1.940454e+00   
min        1.000000e+00       1.000000e+00     1.000000e+01  0.000000e+00   
25%        2.300000e+01       4.000000e+00     1.400000e+01  0.000000e+00   
50%        3.000000e+01       5.000000e+00     2.300000e+01  0.000000e+00   
75%        3.800000e+01       6.000000e+00     5.200000e+01  0.000000e+00   
max        2.857000e+03       4.680000e+02     5.430440e+05  1.109000e+03   

       impressions_last30  clicks_last30  impressions_prev30  clicks_prev30  \
count        2.414248e+06   2.414248e+06        2.414248e+06   2.414248e+06   
mean         2.603523e+01   5.912152e-02        3.115945e+01   6.914327e-02   
std          3.556611e+02   7.117746e-01        4.242729e+02   7.6723

In [ ]:
print(sample_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2414248 entries, 0 to 2414247
Data columns (total 21 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   client_hash_id                 object 
 1   content_hash_id                object 
 2   query_hash_id                  object 
 3   query_char_count               int64  
 4   query_token_count              int64  
 5   window_start                   object 
 6   window_end                     object 
 7   impressions_90d                int64  
 8   clicks_90d                     int64  
 9   impressions_last30             int64  
 10  clicks_last30                  int64  
 11  impressions_prev30             int64  
 12  clicks_prev30                  int64  
 13  avg_position_90d               float64
 14  avg_position_last30            float64
 15  avg_position_prev30            float64
 16  content_total_impressions_90d  int64  
 17  content_visible_query_count    int64  
 18  ra

In [ ]:
print(sample_data.columns)

Index(['client_hash_id', 'content_hash_id', 'query_hash_id',
       'query_char_count', 'query_token_count', 'window_start', 'window_end',
       'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30',
       'impressions_prev30', 'clicks_prev30', 'avg_position_90d',
       'avg_position_last30', 'avg_position_prev30',
       'content_total_impressions_90d', 'content_visible_query_count',
       'rare_query_count', 'rare_impressions_share',
       'anonymized_impressions_share'],
      dtype='object')


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample_data["clicks_greater"] = (sample_data["clicks_last30"]>sample_data["clicks_prev30"])
sample_data["impressions_greater"] = (sample_data["impressions_last30"] > sample_data["impressions_prev30"])
sample_data["avg_position_greater"] = (sample_data["avg_position_last30"]> sample_data["avg_position_prev30"])
sample_data["decline_score"] = (sample_data["clicks_greater"].astype(int)+sample_data["impressions_greater"].astype(int)+sample_data["avg_position_greater"].astype(int))
conditions = [
    sample_data['decline_score'] == 3,
    sample_data['decline_score'] == 2,
    sample_data['decline_score'] == 1,
]
choices = [
    "refresh_now",
    "refresh_soon",
    "monitor",
]
sample_data['refresh_needed'] = np.select(conditions, choices, default="no_action")
'''
Like this the decline_score will be calculated later on. Here i just got important observed signals to compare. Later on will work on more signals.
'''

'\nLike this the decline_score will be calculated later on. Here i just got important observed signals to compare. Later on will work on more signals.\n'

In [ ]:
sample_data['decline_score'].value_counts()

,count
decline_score,
1,1106055
0,993885
2,299415
3,14893


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metrics: F1-Score**<br>
**Reason:**<br>
Since classes are imbalanced so a good macro F1-score as well as individual F1-scores will determine that classes are equally tested. If macro F1-score > 0.70 it will be considered that classes are separated and identified accurately.<br> Good number > 0.70 macro f1-score

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample_data['refresh_needed'].value_counts()

,count
refresh_needed,
monitor,1106055
no_action,993885
refresh_soon,299415
refresh_now,14893


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

***one row = one content item belonging to one client needs refresh or not* **

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
display(sample_data.head(1))

,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share,clicks_greater,impressions_greater,avg_position_greater,decline_score,refresh_needed
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,1466,14,32,0.043656,0.725102,False,False,False,0,no_action


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**if-statement** if contain a lot of conditions, which can be messy and may complicate code as well as decision. It may contain overlapping conditions or make observed signals overcomplicate. Besides, there would be so many conditions that it will take a lot of data analysis, that's why ML beats fixed rule because ML learns the patterns of data and can output accordingly, no need for too much data analysis for covering every edge case.

Besides the rule does not see how much difference is there between these 'clicks_last30' and 'clicks_prev30', 'impressions_last30' and 'impressions_prev30', likewise 'avg_position_last30' and 'avg_position_prev30'. The difference could be of 2 or maybe 200000 but it does not show, on the other hand ML can know this difference, and can learn pattern from it. The rule just check the condition and returns boolean flag - True/False which is converted to 1/0 afterwards.

In [ ]:
# Calculate the actual magnitude of the differences
sample_data['diff_clicks'] = sample_data['clicks_last30'] - sample_data['clicks_prev30']
sample_data['diff_impressions'] = sample_data['impressions_last30'] - sample_data['impressions_prev30']
sample_data['diff_avg_position'] = sample_data['avg_position_last30'] - sample_data['avg_position_prev30']


Clicks max. difference = 426

In [ ]:
sample_data['diff_clicks'].describe()

,diff_clicks
count,2.414248e+06
mean,-1.002175e-02
std,6.478148e-01
min,-1.820000e+02
25%,0.000000e+00
50%,0.000000e+00
75%,0.000000e+00
max,4.260000e+02


Impressions max. difference = 104200

In [ ]:
sample_data['diff_impressions'].describe()

,diff_impressions
count,2.414248e+06
mean,-5.124214e+00
std,3.078797e+02
min,-1.150610e+05
25%,-9.000000e+00
50%,-2.000000e+00
75%,3.000000e+00
max,1.042000e+05


Avg. position difference = 581

In [ ]:
sample_data['diff_avg_position'].describe()

,diff_avg_position
count,1.704767e+06
mean,1.487498e+00
std,1.536074e+01
min,-5.685556e+02
25%,-2.389043e+00
50%,2.000000e-01
75%,4.000000e+00
max,5.815000e+02


In [ ]:
print("Clicks diff, among rows rule marks as declined (clicks_greater == False):")
print(sample_data[sample_data["clicks_greater"] == False]["diff_clicks"].describe())

print("\nImpressions diff, among rows rule marks as declined:")
print(sample_data[sample_data["impressions_greater"] == False]["diff_impressions"].describe())

print("\nAvg position diff, among rows rule marks as declined:")
print(sample_data[sample_data["avg_position_greater"] == True]["diff_avg_position"].describe())

Clicks diff, among rows rule marks as declined (clicks_greater == False):
count    2.349503e+06
mean    -4.856559e-02
std      4.512162e-01
min     -1.820000e+02
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      0.000000e+00
Name: diff_clicks, dtype: float64

Impressions diff, among rows rule marks as declined:
count    1.622701e+06
mean    -2.192615e+01
std      3.090836e+02
min     -1.150610e+05
25%     -1.400000e+01
50%     -5.000000e+00
75%     -2.000000e+00
max      0.000000e+00
Name: diff_impressions, dtype: float64

Avg position diff, among rows rule marks as declined:
count    893272.000000
mean          9.192356
std          14.238672
min           0.000057
25%           1.220492
50%           3.610458
75%          11.166667
max         581.500000
Name: diff_avg_position, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.